# Y-Hat: ¿quién gana la licitación? Ejecutable reproducible

Método: **ranker del historial (LightGBM) + features de texto** (similitud TF-IDF y embeddings MiniLM), sobre un pool de 200 candidatas por licitación. **Modo B (el de la entrega):** el pool se elige con B0 + similitud de texto. **Modo A:** el pool se elige solo con B0; se vuelve a él agregando `--pool-por-b0` a cada comando (o poniendo `POOL_POR_TEXTO_DEFECTO = False` en `modelo_ranker_texto.py`).

Todo se baja del repo y del release de datos: no usa Drive ni archivos de tu compu. **Requiere GPU** (Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU) para calcular los vectores de los contratos.

| Parte | Qué hace | Tiempo aprox. (T4) |
|---|---|---|
| 1 | Prepara el entorno | 3 min |
| 2 | Comprueba el entorno con B0 | 1 min |
| 3 | Valida el método (entrena con lo anterior a 2024, valida 2024S1) | 50 min la primera vez |
| 4 | Ensayo del ejecutable con un test simulado (mismo formato que el test oficial) | 25 min |
| 5 | Resultados para el informe (opcional) | 1 h |
| 6 | Entrega real con el test (modo B y, para tener el otro listo, modo A) | 25 min cada uno |

## Parte 1. Entorno

In [ ]:
# Qué hace: baja el repo, los datos del release y las dos librerías que Colab no trae. No usa Drive.
# Criterio: todo sale de GitHub, así cualquiera puede repetirlo. Versiones fijadas de lightgbm y sentence-transformers.
%cd /content
!test -d grupo_Luabre || git clone -q https://github.com/AbiFrei/grupo_Luabre.git
%cd /content/grupo_Luabre
!test -d data || (curl -sLO https://github.com/bravoivanagustin/ranking-licitaciones/releases/download/datos-v1/data.zip && unzip -oq data.zip && rm -f data.zip)
!pip -q install lightgbm==4.6.0 sentence-transformers==5.7.0
!ls data

In [ ]:
# Qué hace: define lo único que hay que cambiar para otra corrida (verticales, carpeta del test, carpeta de salida) y baja los vectores ya calculados si están publicados.
# Criterio: todas las celdas siguientes leen estas variables. Con VECTORES_URL vacío, los vectores se calculan al correr (más de 1 hora la primera vez en una T4).
import os
VERTICALES = ["rd", "d", "it", "r", "j"]          # para probar rápido: ["rd"]
TEST_RAIZ = "test"                                # carpeta con <vertical>/licitaciones.parquet y empresas.parquet del test (se baja en la Parte 6)
SALIDA_ENTREGA = "entrega"
VECTORES_URL = "https://github.com/AbiFrei/grupo_Luabre/releases/download/vectores-v1"   # vectores MiniLM float32 ya calculados; vacío = se calculan al correr
DATOS = " ".join(f"data/{v}" for v in VERTICALES)
os.makedirs("cache", exist_ok=True)
if VECTORES_URL:
    for V in VERTICALES:
        for nombre in (f"emb_hist_{V}.npy", f"emb_lic_{V}.npz"):
            f = f"cache/{nombre}"
            if not os.path.exists(f):
                !curl -sfL {VECTORES_URL}/{nombre} -o {f} || (rm -f {f}; echo "{nombre}: no está publicado, se calcula al correr")
else:
    print("VECTORES_URL vacío: los vectores se calculan al correr la Parte 3 (más de 1 hora la primera vez).")

In [ ]:
# Qué hace: comprueba la GPU y deja registradas las versiones con las que se obtienen los resultados. Solo lectura.
# Criterio: los resultados del informe se citan con este entorno.
import sys, importlib, torch
assert torch.cuda.is_available(), "Activá la GPU: Entorno de ejecución > Cambiar tipo de entorno de ejecución > T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))
print("python", sys.version.split()[0])
for m in ["numpy", "pandas", "scipy", "sklearn", "lightgbm", "sentence_transformers", "torch", "pyarrow"]:
    print(m, importlib.import_module(m).__version__)

In [ ]:
# Qué hace: deja una "huella" del vocabulario TF-IDF de rd armado de la forma vieja (TfidfVectorizer común) y de la forma nueva (determinista). Solo lectura.
# Criterio: la forma vieja desempata con un argsort inestable y puede dar un vocabulario distinto según la versión de numpy o el procesador.
# Si corrés el notebook en dos sesiones distintas, la huella nueva tiene que ser la misma; la vieja puede cambiar.
import hashlib, numpy as np, pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
import experimento as ex
h = pd.read_parquet("data/rd/historial.parquet")
textos = h.loc[pd.to_datetime(h.fecha) < "2024-01-01", "descripcion"].fillna("")
huella = lambda v: hashlib.md5("\n".join(sorted(v.vocabulary_)).encode()).hexdigest()[:10]
print("CPU:", next(l.split(":")[1].strip() for l in open("/proc/cpuinfo") if l.startswith("model name")))
print("numpy", np.__version__)
viejo = TfidfVectorizer(lowercase=True, stop_words="english", ngram_range=(1, 2), min_df=2, max_features=100_000, sublinear_tf=True).fit(textos)
print("huella forma vieja:", huella(viejo))
print("huella forma nueva:", huella(ex.nuevo_tfidf().fit(textos)))

## Parte 2. Comprobar que el entorno funciona

In [ ]:
# Qué hace: corre el baseline B0 sobre rd y lo evalúa con evaluar.py.
# Criterio: tiene que dar MRR 0,125 en las 328 licitaciones (el número del README). Si da otro, algo del entorno o de los datos cambió.
!python b0.py data/rd --salida b0_rd.csv
!python evaluar.py b0_rd.csv --datos data/rd

## Parte 3. Validar el método

Entrena con las licitaciones anteriores a 2024-01-01 y valida en 2024S1 (alrededor del 20 % más reciente). La primera vez calcula los vectores MiniLM de los contratos de cada vertical y los guarda en `cache/`; las veces siguientes los reutiliza.

In [ ]:
# Qué hace: prueba rápida en la vertical más chica (rd) para ver que todo corre.
# Criterio: solo para chequear; las conclusiones salen de las cinco verticales juntas (celda siguiente).
!python modelo_ranker_texto.py --datos data/rd --salida-dir rankings_validacion

In [ ]:
# Qué hace: entrena y valida el ranker + texto en las cinco verticales y guarda un ranking_<vertical>.csv por cada una.
# Criterio: un modelo por vertical, mismos hiperparámetros en todas. Los vectores de los contratos se calculan una sola vez.
!python modelo_ranker_texto.py --datos {DATOS} --salida-dir rankings_validacion

In [ ]:
# Qué hace: la misma validación con el pool elegido solo con B0 (modo A), para comparar con la celda anterior (modo B).
# Criterio: es la comparación que justifica el modo B. Misma semilla, mismos datos, mismos vectores.
!python modelo_ranker_texto.py --datos {DATOS} --pool-por-b0 --salida-dir rankings_validacion_A

## Parte 4. Ensayo del ejecutable con un test simulado

Simula la entrega: el modelo se entrena solo con licitaciones anteriores a 2024-01-01 y rankea las de 2024S1 como si fueran un test que llega de afuera, con el **mismo formato que el test oficial**: una sola tabla de empresas sin columna `semestre` y sin ganadores. Como el modelo y los datos de entrenamiento son los de la validación, los resultados tienen que dar prácticamente lo mismo.

In [ ]:
# Qué hace: arma sim_train/ (licitaciones y ganadores anteriores a 2024) y sim_test/ (licitaciones de 2024S1 y una sola foto de empresas, la de 2024S1, sin `semestre`).
# Criterio: el historial se copia completo; los filtros por fecha los hace el modelo. sim_test no lleva ganadores y conserva los id originales para poder evaluar.
import pandas as pd, pathlib
CORTE = "2024-01-01"
SIM_DATOS = " ".join(f"sim_train/{v}" for v in VERTICALES)
for V in VERTICALES:
    src = pathlib.Path("data") / V
    lic = pd.read_parquet(src / "licitaciones.parquet")
    gan = pd.read_parquet(src / "ganadores.parquet")
    emp = pd.read_parquet(src / "empresas.parquet")
    hist = pd.read_parquet(src / "historial.parquet")
    tr, te = lic[lic.t < CORTE], lic[lic.t >= CORTE]
    a, b = pathlib.Path("sim_train") / V, pathlib.Path("sim_test") / V
    a.mkdir(parents=True, exist_ok=True); b.mkdir(parents=True, exist_ok=True)
    tr.to_parquet(a / "licitaciones.parquet"); gan[gan.id.isin(tr.id)].to_parquet(a / "ganadores.parquet")
    emp.to_parquet(a / "empresas.parquet"); hist.to_parquet(a / "historial.parquet")
    te.to_parquet(b / "licitaciones.parquet"); emp[emp.semestre == "2024S1"].drop(columns="semestre").drop_duplicates("parent_uei").to_parquet(b / "empresas.parquet")
    print(V, "| entrenamiento:", len(tr), "licitaciones | test simulado:", len(te))

In [ ]:
# Qué hace: corre el modo final igual que se va a correr en la entrega: entrena con sim_train y rankea sim_test.
# Criterio: reutiliza los vectores de cache/ (mismo historial) y codifica solo las licitaciones del test.
!python modelo_ranker_texto.py --datos {SIM_DATOS} --modo final --test-raiz sim_test --salida-dir entrega_sim

In [ ]:
# Qué hace: (1) verifica el formato de cada CSV de la entrega simulada con verificar_entrega.py y (2) lo evalúa contra los ganadores reales de 2024S1.
# Criterio: <V>.csv con id,parent_uei,puesto; hasta 100 por licitación; puestos 1..n; solo empresas de la tabla del test; todas las licitaciones.
import pandas as pd
vs = " ".join(VERTICALES)
!python verificar_entrega.py --csv-dir entrega_sim --test-raiz sim_test --verticales {vs}
for V in VERTICALES:
    csv = f"entrega_sim/{V}.csv"
    gan = pd.read_parquet(f"data/{V}/ganadores.parquet")
    gan[gan.id.isin(pd.read_csv(csv).id)].to_parquet(f"entrega_sim/respuestas_{V}.parquet")
    print(f"=== {V}")
    !python evaluar.py {csv} --datos data/{V} --respuestas entrega_sim/respuestas_{V}.parquet 2>&1 | tail -4

## Parte 5. Resultados para el informe (opcional)

Reproduce las comparaciones del informe con las cinco verticales juntas: B0, B0 + similitud de texto (TF-IDF y embeddings), ranker del historial (sin texto) y ranker + texto.

In [ ]:
# Qué hace: entrena el mismo ranker sin las features de texto (réplica del ranker del historial, que desarrolló otro integrante del grupo).
# Criterio: ablación; la diferencia con el ranker + texto es solo el texto. Usa el mismo código y los mismos vectores.
!python modelo_ranker_texto.py --datos {DATOS} --sin-texto --salida-dir rankings_sin_texto

In [ ]:
# Qué hace: calcula B0 y B0 + similitud de texto (TF-IDF y embeddings) por licitación, con alphas de 0 a 6 (selección 2023, prueba 2024S1).
# Criterio: es el método de nuestra primera etapa; el alpha global es 1,5. Reutiliza los vectores de cache/.
!python experimento_emb.py {' '.join(VERTICALES)} --alphas 0,0.5,1,1.5,2,3,4,6 --etiqueta _ext --salida resultados --cache cache

In [ ]:
# Qué hace: compara B0, B0 + similitud de texto, ranker del historial y ranker + texto, con las cinco verticales juntas (IC95 por bootstrap pareado).
# Criterio: las conclusiones salen de las licitaciones de todas las verticales juntas.
!python comparar_ranker.py --rankings rankings_sin_texto --resultados resultados --rankings-texto rankings_validacion --datos data --verticales {','.join(VERTICALES)}

## Parte 6. Entrega real

El test es el release `test-v1` del repo de la cátedra (`test.zip`); se descomprime en la raíz y deja `test/<vertical>/licitaciones.parquet` y `empresas.parquet`. `empresas` es una sola foto sin columna `semestre`, y no hay historial nuevo: se usa el de `data/`.

In [ ]:
# Qué hace: baja el test oficial y lo descomprime en TEST_RAIZ.
# Criterio: la URL es la del release test-v1 de la cátedra (ver ENTREGA.md). Si cambia, se corrige TEST_URL.
TEST_URL = "https://github.com/bravoivanagustin/ranking-licitaciones/releases/download/test-v1/test.zip"
!test -d test || (curl -sfL {TEST_URL} -o test.zip && unzip -oq test.zip && rm -f test.zip)
!ls {TEST_RAIZ}

In [ ]:
# Qué hace: modo B (el de la entrega): entrena con todas las licitaciones disponibles y escribe <vertical>.csv (rd.csv, d.csv, ...) con id,parent_uei,puesto, hasta 100 empresas por licitación.
# Criterio: reutiliza los vectores de cache/ y codifica solo las licitaciones del test.
!python modelo_ranker_texto.py --datos {DATOS} --modo final --test-raiz {TEST_RAIZ} --salida-dir {SALIDA_ENTREGA}

In [ ]:
# Qué hace: lo mismo en modo A (pool elegido solo con B0), en otra carpeta, para tener los dos listos y elegir al final.
# Criterio: la única diferencia es --pool-por-b0.
!python modelo_ranker_texto.py --datos {DATOS} --pool-por-b0 --modo final --test-raiz {TEST_RAIZ} --salida-dir {SALIDA_ENTREGA}_A

In [ ]:
# Qué hace: verifica los CSV de las dos carpetas contra el test oficial. Solo lectura.
vs = " ".join(VERTICALES)
!python verificar_entrega.py --csv-dir {SALIDA_ENTREGA} --test-raiz {TEST_RAIZ} --verticales {vs}
!python verificar_entrega.py --csv-dir {SALIDA_ENTREGA}_A --test-raiz {TEST_RAIZ} --verticales {vs}

In [ ]:
# Qué hace: muestra la huella (md5) de cada CSV de la entrega y la compara con la de nuestra corrida de referencia, si está cargada. Solo lectura.
# Criterio: con el mismo código, los mismos datos y el mismo tipo de GPU, los CSV tienen que dar exactamente la misma huella. Con otra GPU pueden cambiar
# unos pocos puestos (los vectores del test se calculan en media precisión); si pasa, la huella cambia aunque el resultado sea válido.
import hashlib, torch
REFERENCIA = {}   # {"entrega": {"rd.csv": "<md5>", ...}, "entrega_A": {...}}: huellas de la corrida de referencia del grupo
print("GPU de esta corrida:", torch.cuda.get_device_name(0))
for carpeta in (SALIDA_ENTREGA, SALIDA_ENTREGA + "_A"):
    for V in VERTICALES:
        h = hashlib.md5(open(f"{carpeta}/{V}.csv", "rb").read()).hexdigest()
        ref = REFERENCIA.get(carpeta, {}).get(f"{V}.csv")
        print(f"{carpeta}/{V}.csv  {h}  " + ("" if ref is None else ("IGUAL a la referencia" if h == ref else "DISTINTO de la referencia")))